# SetFit Nível 2 (área responsável)

Treinamento e avaliação de um baseline **SetFit** (contrastive fine-tuning de sentence-transformer + head leve de classificação) para os tickets ITIL de Nível 2 (área técnica responsável). Espelha a metodologia de avaliação dos notebooks `bert_nivel2_training.ipynb` e `svm_nivel2_training.ipynb` para permitir comparação direta no mesmo CV.

- **Input:** `dataset_nivel_2.csv` com colunas `descricao` e `area_responsavel`
- **Modelo:** [SetFit](https://github.com/huggingface/setfit) sobre `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` contrastive fine-tuning com poucos pares + `LogisticRegression` como head de classificação
- **Classes (5):** Aplicações, Servidor, Rede, Segurança, BD
- **Avaliação:** Stratified 5-Fold com predições out-of-fold (mesma metodologia do BERT e do SVM)
- **Output:** modelo em `models/setfit/nivel2/` (zip baixado ao final, no Colab)

> **Por que `descricao` (texto cru) e não `descricao_limpa`?** SetFit usa um sentence-transformer por baixo como o BERT, se beneficia de texto natural (pontuação, acentos, stopwords) em vez do texto normalizado que ajuda o TF-IDF.

> **GPU necessária.** Com 148 exemplos de treino, o SetFit gera ~15k pares contrastivos por treino (default `num_iterations=20`). Em CPU isso é muito lento (testado: ~2h+ por treino); em GPU (T4 do Colab) cada treino deve levar poucos minutos. Este notebook (1 split + 5 folds do CV + modelo final = 7 treinos) foi desenhado para rodar com **Ambiente de execução → GPU** habilitado no Colab.


In [ ]:
!pip install -q setfit==1.1.3 transformers==4.57.6 sentence-transformers==5.6.0 datasets==5.0.0 scikit-learn==1.9.0 accelerate==1.14.0

In [ ]:
from pathlib import Path

import pandas as pd

# Roda tanto localmente (qualquer cwd dentro do repo) quanto no Colab (upload).
CANDIDATOS = [
    Path("data/dataset_nivel_2.csv"),
    Path("../data/dataset_nivel_2.csv"),
    Path(__file__).resolve().parent.parent / "data/dataset_nivel_2.csv"
    if "__file__" in dir()
    else Path("data/dataset_nivel_2.csv"),
]
local = next((c for c in CANDIDATOS if c.exists()), None)

if local is not None:
    df = pd.read_csv(local)
    print(f"Lido localmente: {local}")
else:
    from google.colab import files

    uploaded = files.upload()  # selecione o dataset_nivel_2.csv
    df = pd.read_csv(list(uploaded.keys())[0])

# Compatibilidade: aceita tanto 'area_responsavel' quanto o antigo 'nivel2'
if "area_responsavel" not in df.columns and "nivel2" in df.columns:
    df = df.rename(columns={"nivel2": "area_responsavel"})

print(f"Linhas: {len(df)}")
print(df["area_responsavel"].value_counts())

In [ ]:
import gc
import os

import numpy as np
import torch
from transformers import set_seed

os.environ["WANDB_DISABLED"] = "true"  # evita prompt interativo do wandb no Colab

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TARGET_COL = "area_responsavel"
TEXT_COL = "descricao"  # texto cru sentence-transformer se beneficia do texto natural

df = df.dropna(subset=[TARGET_COL, TEXT_COL]).reset_index(drop=True)
CLASSES = sorted(df[TARGET_COL].unique())
print(f"Classes ({len(CLASSES)}): {CLASSES}")


def free_gpu(*objs):
    for obj in objs:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

## 1. Split único (visão rápida)

Mesmo split do BERT e do SVM (20% estratificado, seed 42). Serve só de referência rápida o número confiável é o do K-Fold abaixo.


In [ ]:
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

X = df[TEXT_COL].tolist()
y = df[TARGET_COL].tolist()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

train_ds = Dataset.from_dict({"text": X_tr, "label": y_tr})

model = SetFitModel.from_pretrained(MODEL_NAME)
args = TrainingArguments(
    batch_size=16, num_epochs=1, seed=RANDOM_SEED, report_to="none"
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()

y_pred = model.predict(X_te)
print(classification_report(y_te, y_pred, labels=CLASSES, zero_division=0))

free_gpu(model, trainer)

## 2. Avaliação robusta Stratified 5-Fold (out-of-fold)

Idêntica ao bloco K-Fold dos notebooks BERT e SVM: cada ticket é predito uma vez (no fold de teste), gerando predições OOF para os 185 tickets. Um `SetFitModel` novo é treinado do zero em cada fold (sem vazamento entre folds).


In [ ]:
import json

from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
EPOCHS_CV = (
    1  # SetFit converge rápido; contrastive pairs já geram muitos exemplos por época
)

X = np.array(df[TEXT_COL].tolist(), dtype=object)
y = np.array(df[TARGET_COL].tolist(), dtype=object)

oof_pred = np.empty(len(df), dtype=object)
fold_f1 = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
    tr_ds = Dataset.from_dict({"text": X[tr_idx].tolist(), "label": y[tr_idx].tolist()})

    fold_model = SetFitModel.from_pretrained(MODEL_NAME)
    fold_args = TrainingArguments(
        batch_size=16, num_epochs=EPOCHS_CV, seed=RANDOM_SEED, report_to="none"
    )
    fold_trainer = Trainer(model=fold_model, args=fold_args, train_dataset=tr_ds)
    fold_trainer.train()

    preds = fold_model.predict(X[te_idx].tolist())
    oof_pred[te_idx] = preds

    f1 = f1_score(y[te_idx], preds, average="macro", zero_division=0)
    fold_f1.append(f1)
    print(f"Fold {fold}: F1 macro = {f1:.3f}")

    free_gpu(fold_model, fold_trainer)

print(f"\nF1 macro por fold: {[round(f, 3) for f in fold_f1]}")
print(f"F1 macro CV (5-fold): {np.mean(fold_f1):.3f} +/- {np.std(fold_f1):.3f}")

print("\n=== Classification report agregado (OOF todos os 185 tickets) ===")
print(
    classification_report(
        y, oof_pred, labels=CLASSES, target_names=CLASSES, zero_division=0
    )
)

print("=== Matriz de confusao (OOF) ===")
print("Classes:", CLASSES)
print(confusion_matrix(y, oof_pred, labels=CLASSES))

cv_results = {
    "model": "setfit+paraphrase-multilingual-MiniLM-L12-v2",
    "n_folds": N_FOLDS,
    "epochs_cv": EPOCHS_CV,
    "text_col": TEXT_COL,
    "f1_macro_per_fold": [round(f, 4) for f in fold_f1],
    "f1_macro_mean": round(float(np.mean(fold_f1)), 4),
    "f1_macro_std": round(float(np.std(fold_f1)), 4),
    "oof_classification_report": classification_report(
        y,
        oof_pred,
        labels=CLASSES,
        target_names=CLASSES,
        output_dict=True,
        zero_division=0,
    ),
}
with open("setfit_nivel2_cv_results.json", "w") as f:
    json.dump(cv_results, f, indent=2, ensure_ascii=False)
print("\nResultados do CV salvos em setfit_nivel2_cv_results.json")

In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(y, oof_pred, labels=CLASSES)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(len(CLASSES)))
ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title("SetFit — Matriz de confusão (OOF, CV 5-fold)")

vmax = cm.max()
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        value = cm[i, j]
        color = "white" if value > vmax / 2 else "black"
        ax.text(j, i, str(value), ha="center", va="center", color=color)

fig.colorbar(im, ax=ax, label="Contagem (OOF)")
fig.tight_layout()
plt.show()

## 3. Modelo final (deploy)

Treina em **todos os 185 tickets** (após o CV validar a abordagem) e salva em `models/setfit/nivel2/`.


In [ ]:
OUTPUT_DIR = "setfit_nivel2"

final_ds = Dataset.from_dict(
    {"text": df[TEXT_COL].tolist(), "label": df[TARGET_COL].tolist()}
)

final_model = SetFitModel.from_pretrained(MODEL_NAME)
final_args = TrainingArguments(
    batch_size=16, num_epochs=1, seed=RANDOM_SEED, report_to="none"
)
final_trainer = Trainer(model=final_model, args=final_args, train_dataset=final_ds)
final_trainer.train()

final_model.save_pretrained(OUTPUT_DIR)

!zip -r {OUTPUT_DIR}.zip {OUTPUT_DIR}/

from google.colab import files  # noqa: E402

files.download(f"{OUTPUT_DIR}.zip")

## Próximos passos

1. Extrair o zip baixado em `models/setfit/nivel2/` na raiz do repo.
2. Comparar o **F1 macro CV** do SetFit com o do BERT (`0.5276 ± 0.1252`) e do SVM (`0.6513 ± 0.0999`, ver `svm_nivel2_cv_results.json` e `notebooks/comparacao_svm_bert.ipynb`).
3. Se o SetFit empatar ou superar o SVM, vale integrar um `SetFitClassifier` no `CascadeClassifier`/`avaliar_modelos.py`, espelhando o `BertClassifier` — isso ficou fora de escopo deste notebook de exploração.
